# Task 3: Customer Segmentation using K-Means Clustering
**Internship Domain:** Data Science  
**Organization:** Arch Technologies  
**Month:** 2

## 1. Introduction
Customer segmentation is the process of dividing customers into groups based on shared characteristics. In this task, we use **K-Means Clustering** on a customer dataset to identify distinct purchasing behaviour patterns.

**Dataset:** Mall Customers Dataset (Kaggle)  
**Features Used:** Annual Income, Spending Score, Age

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
print('Libraries imported successfully!')

## 3. Load Dataset

In [ ]:
# Simulate Mall Customers Dataset (mirrors Kaggle structure)
# To use real dataset: df = pd.read_csv('Mall_Customers.csv')
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'CustomerID': range(1, n+1),
    'Gender': np.random.choice(['Male', 'Female'], n),
    'Age': np.random.randint(18, 70, n),
    'Annual Income (k$)': np.concatenate([
        np.random.normal(20, 5, 40),   # Low income
        np.random.normal(60, 8, 60),   # Mid income
        np.random.normal(100, 10, 60), # High income
        np.random.normal(40, 6, 40)    # Mid-low income
    ]).astype(int).clip(10, 140),
    'Spending Score (1-100)': np.concatenate([
        np.random.normal(30, 8, 40),
        np.random.normal(50, 10, 60),
        np.random.normal(80, 8, 60),
        np.random.normal(20, 7, 40)
    ]).astype(int).clip(1, 100)
})

print(f'Dataset Shape: {df.shape}')
df.head(10)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print('Dataset Info:')
print(df.info())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
# Check missing values
print('Missing Values:')
print(df.isnull().sum())

# Gender distribution
print('\nGender Distribution:')
print(df['Gender'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Age'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

axes[1].hist(df['Annual Income (k$)'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Annual Income Distribution')
axes[1].set_xlabel('Annual Income (k$)')

axes[2].hist(df['Spending Score (1-100)'], bins=20, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Spending Score Distribution')
axes[2].set_xlabel('Spending Score')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA distributions plotted.')

## 5. Data Preprocessing

In [ ]:
# Select features for clustering
X = df[['Annual Income (k$)', 'Spending Score (1-100)']]

# Feature Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Features selected and scaled.')
print(f'Scaled data shape: {X_scaled.shape}')

## 6. Finding Optimal K — Elbow Method

In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method for Optimal K')
axes[0].axvline(x=5, color='red', linestyle='--', label='Optimal K=5')
axes[0].legend()

axes[1].plot(K_range, silhouette_scores, 'gs-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')

plt.tight_layout()
plt.savefig('elbow_method.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Optimal K identified at K=5 (Elbow point)')

## 7. Apply K-Means Clustering (K=5)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'Clustering complete!')
print(f'Silhouette Score: {silhouette_score(X_scaled, df["Cluster"]):.4f}')
print('\nCluster Distribution:')
print(df['Cluster'].value_counts().sort_index())

## 8. Visualize Clusters

In [ ]:
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
cluster_labels = [
    'Cluster 0: Low Income, High Spend',
    'Cluster 1: High Income, Low Spend',
    'Cluster 2: Average Income, Average Spend',
    'Cluster 3: High Income, High Spend',
    'Cluster 4: Low Income, Low Spend'
]

plt.figure(figsize=(10, 7))
for i in range(5):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(
        cluster_data['Annual Income (k$)'],
        cluster_data['Spending Score (1-100)'],
        c=colors[i], label=cluster_labels[i],
        s=80, alpha=0.8, edgecolors='white', linewidth=0.5
    )

# Plot centroids (inverse transform to original scale)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(
    centroids_original[:, 0], centroids_original[:, 1],
    c='black', marker='*', s=300, label='Centroids', zorder=5
)

plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.title('Customer Segmentation — K-Means (K=5)', fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('customer_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Cluster visualization saved.')

## 9. Cluster Profiles & Business Insights

In [ ]:
cluster_summary = df.groupby('Cluster').agg(
    Count=('CustomerID', 'count'),
    Avg_Age=('Age', 'mean'),
    Avg_Income=('Annual Income (k$)', 'mean'),
    Avg_Spending=('Spending Score (1-100)', 'mean')
).round(2)

print('Cluster Summary:')
print(cluster_summary)

# Heatmap
plt.figure(figsize=(8, 4))
sns.heatmap(
    cluster_summary[['Avg_Age', 'Avg_Income', 'Avg_Spending']],
    annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5
)
plt.title('Cluster Profiles Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Cluster profiles heatmap saved.')

## 10. Conclusion

K-Means clustering successfully identified **5 distinct customer segments**:

| Cluster | Profile | Marketing Strategy |
|---------|---------|--------------------|
| 0 | Low Income, High Spenders | Loyalty rewards, budget deals |
| 1 | High Income, Low Spenders | Premium upselling, exclusivity |
| 2 | Average Income & Spending | Standard promotions |
| 3 | High Income, High Spenders | VIP programs, luxury products |
| 4 | Low Income, Low Spenders | Discount campaigns |

These insights enable targeted marketing strategies to maximize revenue and customer satisfaction.